In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
def load_prices(file_path, date_column='Date', price_column='Close', ticker_column='Ticker'):
    """
    Загрузка данных о ценах из файла
    
    Поддерживаемые форматы:
    1. Wide format: каждая колонка - тикер, индекс - дата
    2. Long format: колонки с датой, тикером и ценой
    
    Параметры:
    - file_path: путь к файлу (CSV, Excel)
    - date_column: название колонки с датой (для long format)
    - price_column: название колонки с ценой (для long format)
    - ticker_column: название колонки с тикером (для long format)
    
    Возвращает:
    - DataFrame с ценами (индекс - дата, колонки - тикеры)
    """
    # Определяем формат файла по расширению
    if file_path.endswith('.csv'):
        data = pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        data = pd.read_excel(file_path)
    else:
        raise ValueError("Неподдерживаемый формат файла. Используйте CSV или Excel.")
    
    # Определяем формат данных
    # Если есть колонка с датой и только одна колонка с ценами - вероятно long format
    if date_column in data.columns and price_column in data.columns:
        # Long format
        print("Обнаружен long format данных")
        data[date_column] = pd.to_datetime(data[date_column])
        
        # Преобразуем в wide format
        prices = data.pivot(index=date_column, columns=ticker_column, values=price_column)
        
    else:
        # Предполагаем wide format
        print("Обнаружен wide format данных")
        # Если первый столбец - дата, устанавливаем его как индекс
        if data.columns[0] in ['Date', 'date', 'Дата'] or pd.api.types.is_datetime64_any_dtype(data.iloc[:, 0]):
            data.set_index(data.columns[0], inplace=True)
            data.index = pd.to_datetime(data.index)
        
        prices = data
    
    # Сортируем по дате
    prices = prices.sort_index()
    
    # Убираем строки с полностью пропущенными значениями
    prices = prices.dropna(how='all')
    
    # Убираем колонки с полностью пропущенными значениями
    prices = prices.dropna(axis=1, how='all')
    
    print(f"Загружено {len(prices.columns)} акций, {len(prices)} дней данных")
    print(f"Период: {prices.index[0].strftime('%Y-%m-%d')} - {prices.index[-1].strftime('%Y-%m-%d')}")
    
    return prices

In [3]:
load_prices("data/prices_moex_new.csv")

ParserError: Error tokenizing data. C error: Expected 20 fields in line 3, saw 22
